## Example 1: Basic HTTP connection pool usage

In [ ]:
import httpcore

# Example 1: Basic HTTP connection pool usage
print("Example 1: HTTP Connection Pool")

# Create connection pool
pool = httpcore.ConnectionPool()
print(f"Connection pool created: {type(pool).__name__}")

# Create HTTP request
request = httpcore.Request(
    method=b"GET",
    url=b"https://httpbingo.org/get",
    headers=[(b"user-agent", b"httpcore-test")]
)

print(f"Request created: {request.method.decode()} {request.url}")
print(f"Request headers: {len(request.headers)} headers")

# Show request details
print(f"HTTP method: {request.method}")
print(f"Target URL: {request.url}")

# Requests normalise their URL into scheme, host, port and target
assert request.method == b"GET"
assert request.url.scheme == b"https" and request.url.host == b"httpbingo.org"
assert request.url.target == b"/get" and request.url.port is None
assert request.headers == [(b"user-agent", b"httpcore-test")]
# A str method and URL are encoded to bytes
assert httpcore.Request("GET", "https://example.com/path").method == b"GET"
# A new pool holds no connections until a request is sent
assert pool.connections == []
pool.close()

## Example 2: Async connection pool usage

In [ ]:
import httpcore
import asyncio

# Example 2: Async connection pool usage
print("Example 2: Async Connection Pool")

async def test_async_pool():
    # Create async connection pool
    async_pool = httpcore.AsyncConnectionPool()
    print(f"Async pool created: {type(async_pool).__name__}")
    
    # Create request
    request = httpcore.Request(
        method=b"GET",
        url=b"https://httpbingo.org/json"
    )
    
    print(f"Async request prepared: {request.method.decode()}")
    print(f"Target: {request.url}")
    
    # Show pool info
    print(f"Pool connections: {len(async_pool.connections)}")
    assert async_pool.connections == []
    assert request.url.target == b"/json"

    await async_pool.aclose()
    print("Async pool closed")

# Run async example
await test_async_pool()
print("Async connection pool example completed successfully")

## Example 3: HTTP Origin and URL handling

In [ ]:
import httpcore

# Example 3: HTTP Origin and URL handling
print("Example 3: HTTP Origin and URL handling")

# Create HTTP origin
origin = httpcore.Origin(b"https", b"httpbingo.org", 443)
print(f"Origin created: {type(origin).__name__}")
print(f"Scheme: {origin.scheme.decode()}")
print(f"Host: {origin.host.decode()}")
print(f"Port: {origin.port}")

# Create URL from components
url = httpcore.URL(b"https://httpbingo.org/post")
print(f"URL created: {type(url).__name__}")
print(f"Full URL: {url}")

# Create a request with detailed headers
request = httpcore.Request(
    method=b"POST",
    url=url,
    headers=[
        (b"content-type", b"application/json"),
        (b"user-agent", b"httpcore-client"),
        (b"accept", b"application/json")
    ]
)

print(f"Request method: {request.method.decode()}")
print(f"Request URL: {request.url}")
print(f"Headers count: {len(request.headers)}")

assert origin.scheme == b"https" and origin.host == b"httpbingo.org" and origin.port == 443
# The URL's origin fills in the default port for its scheme
assert url.origin == origin
assert request.method == b"POST" and len(request.headers) == 3
# A URL can also be built from its parts
built = httpcore.URL(scheme=b"http", host=b"example.com", port=8080, target=b"/a?b=1")
assert bytes(built) == b"http://example.com:8080/a?b=1"

## Example 4: HTTP headers and proxy configuration

In [ ]:
import httpcore

# Example 4: HTTP headers and proxy configuration
print("Example 4: HTTP headers and proxy configuration")

# Create headers for HTTP/2 style request
headers = [
    (b"user-agent", b"httpcore-client/1.0"),
    (b"accept", b"application/json"),
    (b"accept-encoding", b"gzip, deflate"),
    (b"connection", b"keep-alive")
]

print(f"Headers created: {len(headers)} headers")
for name, value in headers:
    print(f"  {name.decode()}: {value.decode()}")

# Create proxy configuration
proxy = httpcore.Proxy(b"http://proxy.example.com:8080")
print(f"Proxy configured: {type(proxy).__name__}")
print(f"Proxy URL: {proxy.url}")

# Create request with proxy headers
request = httpcore.Request(
    method=b"GET",
    url=b"https://httpbingo.org/headers",
    headers=headers
)

print(f"Request with proxy headers:")
print(f"  Method: {request.method.decode()}")
print(f"  URL: {request.url}")
print(f"  Total headers: {len(request.headers)}")

assert proxy.url.scheme == b"http" and proxy.url.host == b"proxy.example.com" and proxy.url.port == 8080
assert request.headers == headers
# Proxy credentials become a Proxy-Authorization header (Basic base64 of user:pw)
auth_proxy = httpcore.Proxy("http://proxy.example.com:8080", auth=("user", "pw"))
assert auth_proxy.headers == [(b"Proxy-Authorization", b"Basic dXNlcjpwdw==")]

## Example 5: Response handling and status codes

In [ ]:
import httpcore

# Example 5: Response handling and status codes
print("Example 5: Response handling and status codes")

# Create response object manually (simulating received response)
response = httpcore.Response(
    status=200,
    headers=[
        (b"content-type", b"application/json"),
        (b"server", b"nginx/1.18.0"),
        (b"content-length", b"25")
    ],
    extensions={"http_version": b"HTTP/1.1"}
)

print(f"Response created: {type(response).__name__}")
print(f"Status code: {response.status}")
print(f"HTTP version: {response.extensions['http_version'].decode()}")

# Process response headers
headers_dict = dict(response.headers)
print(f"Response headers: {len(response.headers)} headers")
print(f"Content type: {headers_dict.get(b'content-type', b'').decode()}")
print(f"Server: {headers_dict.get(b'server', b'').decode()}")

# Demonstrate different status codes
status_codes = [200, 404, 500, 302]
for code in status_codes:
    test_response = httpcore.Response(status=code, headers=[])
    print(f"Status {code}: {test_response.status}")

assert response.status == 200
assert headers_dict[b"content-type"] == b"application/json"
assert response.extensions["http_version"] == b"HTTP/1.1"
# A response built with a body returns it from read()
body_response = httpcore.Response(200, content=b'{"ok": true}')
assert body_response.read() == b'{"ok": true}' and body_response.content == b'{"ok": true}'